In [ ]:
import os

# Check what files are available
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/andrewmvd/trip-advisor-hotel-reviews/tripadvisor_hotel_reviews.csv')

# Basic exploration
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Rating distribution
print("Rating Distribution:")
print(df['Rating'].value_counts().sort_index())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSample reviews per rating:")
for rating in [1, 3, 5]:
    print(f"\nRating {rating}:")
    print(df[df['Rating']==rating]['Review'].iloc[0])

In [ ]:
from textblob import TextBlob

# Add sentiment scores
df['polarity'] = df['Review'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['subjectivity'] = df['Review'].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)

# Add sentiment label
def get_sentiment(polarity):
    if polarity > 0.1:
        return 'Positive'
    elif polarity < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment'] = df['polarity'].apply(get_sentiment)

print("Sentiment Distribution:")
print(df['Sentiment'].value_counts())

print("\nAverage polarity per rating:")
print(df.groupby('Rating')['polarity'].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1 - Rating distribution
df['Rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
)
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Plot 2 - Sentiment distribution
df['Sentiment'].value_counts().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=['#2ecc71','#95a5a6','#e74c3c']
)
axes[1].set_title('Sentiment Distribution')

# Plot 3 - Polarity by Rating
df.groupby('Rating')['polarity'].mean().plot(
    kind='bar', ax=axes[2], color='steelblue'
)
axes[2].set_title('Average Polarity by Rating')
axes[2].set_xlabel('Rating')
axes[2].set_ylabel('Polarity')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Prepare text
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english', max_features=1000)
dtm = vectorizer.fit_transform(df['Review'])

# LDA model
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(dtm)

# Print top words per topic
print("Top 10 words per topic:\n")
for i, topic in enumerate(lda.components_):
    top_words = [vectorizer.get_feature_names_out()[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i+1}: {', '.join(top_words)}")

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Positive reviews wordcloud
positive_text = ' '.join(df[df['Sentiment']=='Positive']['Review'].tolist())
wc_pos = WordCloud(width=800, height=400, background_color='white', 
                   colormap='Greens').generate(positive_text)
axes[0].imshow(wc_pos)
axes[0].set_title('Positive Reviews - Common Words', fontsize=14)
axes[0].axis('off')

# Negative reviews wordcloud
negative_text = ' '.join(df[df['Sentiment']=='Negative']['Review'].tolist())
wc_neg = WordCloud(width=800, height=400, background_color='white',
                   colormap='Reds').generate(negative_text)
axes[1].imshow(wc_neg)
axes[1].set_title('Negative Reviews - Common Words', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Prepare data
X = df['Review']
y = df['Rating']

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix style heatmap
sentiment_rating = pd.crosstab(df['Rating'], df['Sentiment'])
plt.figure(figsize=(8,5))
sns.heatmap(sentiment_rating, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Rating vs Sentiment Heatmap')
plt.show()

In [ ]:
df['review_length'] = df['Review'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(12,4))
for rating in [1,2,3,4,5]:
    subset = df[df['Rating']==rating]['review_length']
    plt.hist(subset, alpha=0.5, label=f'Rating {rating}', bins=30)
plt.xlabel('Review Length (words)')
plt.ylabel('Count')
plt.title('Review Length Distribution by Rating')
plt.legend()
plt.show()

print("Average review length per rating:")
print(df.groupby('Rating')['review_length'].mean().round(1))

In [ ]:
# Assign dominant topic to each review
topic_names = {
    0: 'Room & Service',
    1: 'Location & Experience', 
    2: 'Resort & Amenities',
    3: 'Cleanliness & Breakfast',
    4: 'Beach & Activities'
}

df['dominant_topic'] = lda.transform(dtm).argmax(axis=1)
df['topic_name'] = df['dominant_topic'].map(topic_names)

# Topic distribution
plt.figure(figsize=(10,5))
df['topic_name'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Review Distribution by Topic')
plt.xlabel('Topic')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nAverage rating per topic:")
print(df.groupby('topic_name')['Rating'].mean().round(2))

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
df['polarity'].hist(bins=50, color='steelblue', edgecolor='black')
plt.title('Polarity Distribution')
plt.xlabel('Polarity Score')
plt.ylabel('Count')

plt.subplot(1,2,2)
df.groupby('Rating')['subjectivity'].mean().plot(kind='bar', color='coral')
plt.title('Average Subjectivity by Rating')
plt.xlabel('Rating')
plt.ylabel('Subjectivity')

plt.tight_layout()
plt.show()

In [ ]:
print("="*50)
print("TOURIST SENTIMENT ANALYSIS - SUMMARY")
print("="*50)
print(f"\nTotal Reviews Analyzed: {len(df):,}")
print(f"\nSentiment Breakdown:")
print(df['Sentiment'].value_counts().to_string())
print(f"\nAverage Rating: {df['Rating'].mean():.2f}/5")
print(f"Average Polarity: {df['polarity'].mean():.3f}")
print(f"Average Review Length: {df['review_length'].mean():.0f} words")
print(f"\nMost Common Topic: {df['topic_name'].value_counts().index[0]}")
print(f"\nHighest Rated Topic:")
print(df.groupby('topic_name')['Rating'].mean().sort_values(ascending=False).head(1))
print(f"\nLowest Rated Topic:")
print(df.groupby('topic_name')['Rating'].mean().sort_values().head(1))
print("="*50)

In [ ]:
# Save enriched dataset
df.to_csv('/kaggle/working/sentiment_analysis_results.csv', index=False)
print("Results saved!")
print("\nFinal dataframe columns:")
print(df.columns.tolist())
print("\nShape:", df.shape)